# StageBridge

Thin orchestration notebook for Mission 2.


## 1. Load Config

Compose the active repo config and keep all implementation logic in the package.


In [ ]:
from pathlib import Path

import pandas as pd

from stagebridge.notebook_api import compose_config
from stagebridge.pipelines.run_full import run_full
from stagebridge.results import (
    ensure_registry_files,
    load_current_scratch_run,
    promote_current_scratch_run,
    read_promoted_results,
    read_results_registry,
    write_scratch_run,
)

cfg = compose_config("default", overrides=["data=local", "train=smoke", "evaluation=baseline"])
cfg


## 2. Validate Environment Or Run Context

Create the registry files and confirm the scratch workspace root.


In [ ]:
ensure_registry_files()
print("Scratch root:", Path("outputs/scratch/current"))
print("Registry root:", Path("results/registry"))


## 3. Run Pipeline Entry Point(s)

Use the package pipeline namespace only.


In [ ]:
pipeline_output = run_full(cfg)
pipeline_output


## 4. Display Outputs

Summarize the package return payload without introducing notebook-side business logic.


In [ ]:
step_rows = [
    {"step": step_name, "ok": step_payload.get("ok", False), "status": step_payload.get("status", "n/a")}
    for step_name, step_payload in pipeline_output.get("steps", {}).items()
]
pd.DataFrame(step_rows)


## 5. Write Scratch Run Record

Ordinary runs go to `outputs/scratch/current/` only.


In [ ]:
scratch_run = write_scratch_run(
    cfg,
    pipeline_output,
    experiment_name="mission2_smoke",
    mode="smoke_infrastructure",
    status="complete",
    notebook_source="StageBridge.ipynb",
    milestone_candidate=False,
    next_recommended_step="Promote only if you want to keep this infrastructure smoke artifact.",
)
scratch_run


## 6. Inspect Registry

Review the durable lightweight registry and the current scratch payload.


In [ ]:
registry_df = pd.DataFrame(read_results_registry())
registry_df.tail(5) if not registry_df.empty else registry_df


In [ ]:
current_scratch = load_current_scratch_run()
promoted_results = read_promoted_results()
current_scratch, promoted_results


## 7. Optionally Promote Milestone

Promotion is explicit and milestone-only.


In [ ]:
# Example:
# promote_current_scratch_run(
#     milestone_id="mission2_smoke_keep",
#     summary="Infrastructure smoke run for Mission 2 results-system validation.",
#     importance_level="smoke",
#     interpretation_notes="This confirms scratch writing, registry updates, and milestone promotion.",
#     next_step_recommendation="Proceed to Mission 3 scientific implementation only after results plumbing remains stable.",
# )
